In [1]:
from pathlib import Path
import os

# Make sure we are in the root directory
def set_project_root(marker="pyproject.toml"):
    path = Path.cwd()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            os.chdir(parent)
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")
set_project_root()

PosixPath('/home/coder/synthetic-data-bootcamp')

In [2]:
import pandas as pd
from implementations.tabular_data.utils import download_and_save_data

In [3]:
ROOT = Path.cwd()
IMPLEMENTATION_ROOT = ROOT / "implementations" / "tabular_data"
print("IMPLEMENTATION ROOT: ", IMPLEMENTATION_ROOT)
# Set data and output directories
base_data_dir = IMPLEMENTATION_ROOT / "single_table" / "data"
base_output_dir = IMPLEMENTATION_ROOT / "single_table" / "results"

# Default dataset
DATASET_NAME = "Berka" #More details about the Berka data: https://webpages.charlotte.edu/mirsad/itcs6265/group1/domain.html 
TABLE_NAME = "trans" # Transaction table

IMPLEMENTATION ROOT:  /home/coder/synthetic-data-bootcamp/implementations/tabular_data


In [4]:
# Download the raw dataset. 
download_and_save_data(DATASET_NAME, Path(base_data_dir,"raw_data"))

INFO:implementations.tabular_data.utils:Downloading the transaction table of theBerka dataset from https://drive.google.com/drive/folders/1LA23XSQTin7p6oWtxg7GL7_Qm_sSJ4sn -> /home/coder/synthetic-data-bootcamp/implementations/tabular_data/single_table/data/raw_data
INFO:implementations.tabular_data.utils:Downloading URL: https://drive.google.com/drive/folders/1LA23XSQTin7p6oWtxg7GL7_Qm_sSJ4sn -> /home/coder/synthetic-data-bootcamp/implementations/tabular_data/single_table/data/raw_data
Retrieving folder contents


Processing file 1pEVaM5iuyqK1YSizBZ4MkMBI6ot0kr3V trans.csv


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1pEVaM5iuyqK1YSizBZ4MkMBI6ot0kr3V
To: /home/coder/synthetic-data-bootcamp/implementations/tabular_data/single_table/data/raw_data/trans.csv
100%|██████████| 69.4M/69.4M [00:01<00:00, 64.4MB/s]
Download completed


PosixPath('/home/coder/synthetic-data-bootcamp/implementations/tabular_data/single_table/data/raw_data')

In [5]:
# Load the raw data
raw_data = pd.read_csv(Path(base_data_dir,"raw_data",f"{TABLE_NAME}.csv"))
print(f"Table size: {raw_data.shape}")
print(raw_data.head())

Table size: (1056320, 1)
  trans_id;"account_id";"date";"type";"operation";"amount";"balance";"k_symbol";"bank";"account"
0  695247;2378;930101;"PRIJEM";"VKLAD";700.00;700...                                            
1  171812;576;930101;"PRIJEM";"VKLAD";900.00;900....                                            
2  207264;704;930101;"PRIJEM";"VKLAD";1000.00;100...                                            
3  1117247;3818;930101;"PRIJEM";"VKLAD";600.00;60...                                            
4  579373;1972;930102;"PRIJEM";"VKLAD";400.00;400...                                            


## Preprocessing step

This step turns raw tabular data into the train/holdout CSVs and metadata files used by the training and evaluation pipelines.

### Input

Raw data at the configured input path (for Berka, typically `trans.csv` or a asc export). Separators can differ by source (e.g. `;` for `.asc`, `,` for CSV)—detect or set the delimiter explicitly so columns load correctly.

### Optional sampling

If the dataset is large, or you want to try smaller sizes, subsample before or after encoding. You also choose whether to fit label encoders (and build domain sizes) on the **full** table or only the **sampled** portion:

- **Fit on full data, then sample** — domain sizes and category codes stay closer to a full-population export.
- **Sample first, then fit** — faster on larger tables, but unique counts and encodings can differ from a full-data run.

### Outputs

Written under the data directory (e.g. `single_table/data/`):

| File | Role |
|------|------|
| `{table_name}.csv` | Train split used to fit the diffusion model |
| `{table_name}_holdout.csv` | Holdout split reserved for evaluation (not used in training) |
| `{table_name}_domain.json` | Per-column `size` (nunique) and `type` (`continuous` / `discrete`) |
| `dataset_meta.json` | Table graph: tables and parent/child relations (single-table: one table, no parents) |
| `meta_info.json` | Evaluation schema: numeric/categorical column indices, target column, and task type (e.g. `regression`, `binary_classification`) |

Optional artifacts (when saved): `preprocess_meta.json` and label-encoder pickles for inverse transforms.

### Typical processing steps

Adjust as needed for your dataset; for Berka `trans.csv` these are the usual steps:

1. **Drop ID columns** that should not be synthesized (e.g. `trans_id`, `account_id`).
2. **Clean / engineer features** — fill or drop missing values; convert dates; rename columns if required (Berka: `date` → day offsets as `trans_date`, `type` → `trans_type`).
3. **Encode categoricals** (e.g. with `LabelEncoder` converts categorical values to integers between `0` and `n_classes-1`) and leave continuous columns numeric.
4. **Optionally subsample**, then **split** into train / holdout.
5. **Write** the CSV splits and the three metadata JSON files above.


In [6]:
if DATASET_NAME == "Berka":
    from implementations.tabular_data.single_table.data_processing.preprocess_berka_trans import preprocess_berka_trans

    preprocess_berka_trans(
        input_path=Path(base_data_dir,"raw_data",f"{TABLE_NAME}.csv"),
        output_dir=base_data_dir, # Save the processed data in the base data directory to be used in training and evaluation notebooks
        sample_size=20000, # The original dataset is too large, consider sub-sampling to 50,000 rows.
        holdout_ratio=0.2, # This portion of the sample size will be used as holdout set.
        seed=42,
        save_artifacts=False,# You can enable this to save not necessary artifacts such as label encoders and processing meta information.
    )
    print(f"Preprocessed data saved to {base_data_dir}")
else:
    # TODO: add pre-processing for your dataset
    raise ValueError(f"Dataset {DATASET_NAME} not supported. Implement your own dataset preprocessing function.")

Loading /home/coder/synthetic-data-bootcamp/implementations/tabular_data/single_table/data/raw_data/trans.csv (sep=';') ...
Loaded 1,056,320 rows, ['trans_id', 'account_id', 'date', 'type', 'operation', 'amount', 'balance', 'k_symbol', 'bank', 'account']
Sampled to 20,000 rows before encoding.
Date epoch (YYMMDD): 930116  →  trans_date = days since that date
LabelEncoder classes:
  trans_type: ['PRIJEM', 'VYBER', 'VYDAJ']
  operation: ['', 'PREVOD NA UCET', 'PREVOD Z UCTU', 'VKLAD', 'VYBER', 'VYBER KARTOU']
  k_symbol: ['', ' ', 'DUCHOD', 'POJISTNE', 'SANKC. UROK', 'SIPO', 'SLUZBY', 'UROK', 'UVER']
  bank: ['', 'AB', 'CD', 'EF', 'GH', 'IJ', 'KL', 'MN', 'OP', 'QR', 'ST', 'UV', 'WX', 'YZ']
Split: train=16,000  holdout=4,000  (holdout_ratio=0.2)
Wrote train data to /home/coder/synthetic-data-bootcamp/implementations/tabular_data/single_table/data/trans.csv
Wrote holdout data to /home/coder/synthetic-data-bootcamp/implementations/tabular_data/single_table/data/trans_holdout.csv
Wrote /home